# TomatoSense — Web App (Frontend UI)
### IT2011 — Simple frontend to check the trained sentiment model



## 0. Setup



In [12]:
!pip install -q datasets transformers torch flask pyngrok

import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")
print("Dataset loaded:", ds)


Dataset loaded: DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})


## 1. Fine-Tune DistilBERT (the Group's Selected Best Model)



In [13]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=64)

train_tok = ds["train"].map(tokenize, batched=True)

distilbert_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

training_args = TrainingArguments(
    output_dir="./webapp_model",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    logging_steps=200,
    report_to="none",
)
trainer = Trainer(model=distilbert_model, args=training_args, train_dataset=train_tok)
trainer.train()

distilbert_model.eval()
print("DistilBERT fine-tuned and ready -- this is the only model the app will use.")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
200,0.463319
400,0.395583
600,0.315929
800,0.161587
1000,0.195178


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DistilBERT fine-tuned and ready -- this is the only model the app will use.


## 2. Prediction Function



In [14]:
def predict_sentiment(review_text):
    inputs = tokenizer(review_text, return_tensors="pt", truncation=True, padding=True, max_length=64)
    inputs = {k: v.to(distilbert_model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = distilbert_model(**inputs)
    pred = torch.argmax(outputs.logits, dim=1).item()
    proba = torch.softmax(outputs.logits, dim=1)[0]
    return {
        "label": "Positive" if pred == 1 else "Negative",
        "confidence": round(float(max(proba).item()) * 100, 1),
    }

# Quick self-test
print(predict_sentiment("This film was a masterpiece from start to finish."))


{'label': 'Positive', 'confidence': 99.7}


## 3. The Web App (Flask + Custom Frontend)


In [15]:
from flask import Flask, request, jsonify, Response

app = Flask(__name__)

PAGE_HTML = """
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>TomatoSense — Sentiment Checker</title>
<style>
  :root {
    --wine: #531A22; --gold: #D8A838; --gold-deep: #B9862A;
    --green: #2E9E6D; --red: #C1272D; --cream: #F4F1EA;
    --body-grey: #5B5147; --card: #FFFFFF;
  }
  * { box-sizing: border-box; }
  body {
    margin: 0; font-family: -apple-system, Segoe UI, Roboto, Arial, sans-serif;
    background: var(--cream); color: var(--wine); min-height: 100vh;
    display: flex; align-items: center; justify-content: center; padding: 24px;
  }
  .card {
    background: var(--card); max-width: 560px; width: 100%;
    border-radius: 16px; box-shadow: 0 10px 40px rgba(83,26,34,0.12);
    padding: 36px 36px 28px; border: 1px solid #E4DDD1;
  }
  .brand { font-size: 13px; font-weight: 700; letter-spacing: 2px; color: var(--gold-deep); text-transform: uppercase; }
  h1 { margin: 6px 0 4px; font-size: 26px; }
  .subtitle { color: var(--body-grey); font-size: 14px; margin-bottom: 6px; }
  .model-tag {
    display: inline-block; background: #FBEEE9; color: var(--gold-deep); font-size: 11.5px;
    font-weight: 700; padding: 4px 12px; border-radius: 20px; margin-bottom: 20px; letter-spacing: 0.3px;
  }
  textarea {
    width: 100%; min-height: 110px; border-radius: 10px; border: 1.5px solid #E4DDD1;
    padding: 14px; font-size: 15px; font-family: inherit; resize: vertical; color: var(--wine);
  }
  textarea:focus { outline: none; border-color: var(--gold); }
  button {
    margin-top: 14px; background: var(--wine); color: white; border: none;
    padding: 12px 26px; border-radius: 8px; font-size: 15px; font-weight: 600;
    cursor: pointer; transition: background 0.15s;
  }
  button:hover { background: #6E2530; }
  button:disabled { opacity: 0.6; cursor: not-allowed; }
  .examples { margin-top: 10px; font-size: 12.5px; color: var(--body-grey); }
  .examples a { color: var(--gold-deep); cursor: pointer; text-decoration: underline; margin-right: 10px; }
  .result { margin-top: 28px; display: none; text-align: center; padding: 24px; border-radius: 12px; background: #FAF7F1; border: 1px solid #E4DDD1; }
  .badge {
    display: inline-block; padding: 10px 28px; border-radius: 24px; font-weight: 700; font-size: 20px; color: white;
  }
  .badge.positive { background: var(--green); }
  .badge.negative { background: var(--red); }
  .confidence { font-size: 13.5px; color: var(--body-grey); margin-top: 12px; }
  .loading { font-size: 14px; color: var(--body-grey); margin-top: 18px; display: none; }
  .error { color: var(--red); font-size: 13.5px; margin-top: 12px; display: none; }
</style>
</head>
<body>
  <div class="card">
    <div class="brand">TomatoSense</div>
    <h1>Movie Review Sentiment Checker</h1>
    <div class="subtitle">Type a movie review below to check its sentiment.</div>
    <div class="model-tag">Model: DistilBERT — 83.8% test accuracy (group's selected final model)</div>

    <textarea id="reviewInput" placeholder="e.g. This film was a masterpiece from start to finish."></textarea>
    <div class="examples">
      Try: <a onclick="fillExample(0)">a positive review</a><a onclick="fillExample(1)">a negative review</a><a onclick="fillExample(2)">a tricky mixed review</a>
    </div>
    <br>
    <button id="submitBtn" onclick="checkSentiment()">Check Sentiment</button>

    <div class="loading" id="loading">Running model…</div>
    <div class="error" id="error"></div>

    <div class="result" id="result">
      <span class="badge" id="badge"></span>
      <div class="confidence" id="conf"></div>
    </div>
  </div>

<script>
const examples = [
  "An absolute masterpiece. The acting was brilliant and the plot kept me on the edge of my seat.",
  "A tedious, poorly written mess that wastes its talented cast.",
  "The visuals are gorgeous, but the story is a complete disaster."
];
function fillExample(i) { document.getElementById("reviewInput").value = examples[i]; }

async function checkSentiment() {
  const text = document.getElementById("reviewInput").value.trim();
  const btn = document.getElementById("submitBtn");
  const loading = document.getElementById("loading");
  const result = document.getElementById("result");
  const errorBox = document.getElementById("error");
  errorBox.style.display = "none";

  if (!text) { errorBox.textContent = "Please type a review first."; errorBox.style.display = "block"; return; }

  btn.disabled = true; loading.style.display = "block"; result.style.display = "none";

  try {
    const resp = await fetch("/predict", {
      method: "POST", headers: { "Content-Type": "application/json" },
      body: JSON.stringify({ text })
    });
    if (!resp.ok) throw new Error("Server error");
    const data = await resp.json();

    const badge = document.getElementById("badge");
    badge.textContent = data.label;
    badge.className = "badge " + data.label.toLowerCase();
    document.getElementById("conf").textContent = "Confidence: " + data.confidence + "%";
    result.style.display = "block";
  } catch (e) {
    errorBox.textContent = "Something went wrong reaching the model. Check the Colab cell is still running.";
    errorBox.style.display = "block";
  } finally {
    btn.disabled = false; loading.style.display = "none";
  }
}
</script>
</body>
</html>
"""

@app.route("/")
def home():
    return Response(PAGE_HTML, mimetype="text/html")

@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json()
    text = data.get("text", "")
    if not text.strip():
        return jsonify({"error": "empty input"}), 400
    return jsonify(predict_sentiment(text))

print("Flask app defined.")


Flask app defined.


## 4. Launch the App (Public Link)


In [16]:
from pyngrok import ngrok

NGROK_TOKEN = "3GzkrQuBxVyBkEDZ8g9mvI0UP4I_3J1fp2orbwDk8mrTUF7g"
ngrok.set_auth_token(NGROK_TOKEN)

public_url = ngrok.connect(5000)
print("Your TomatoSense app is live at:", public_url)

app.run(port=5000)


Your TomatoSense app is live at: NgrokTunnel: "https://spherical-desktop-startle.ngrok-free.dev" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [06/Oct/2026 04:31:36] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [06/Oct/2026 04:31:40] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [06/Oct/2026 04:31:43] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [06/Oct/2026 04:31:46] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [06/Oct/2026 04:31:48] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [06/Oct/2026 04:31:49] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [06/Oct/2026 04:31:51] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [06/Oct/2026 04:31:55] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [06/Oct/2026 04:32:05] "POST /predict HTTP/1.1" 200 -
